# 🦶 piediabetico.lat — Pipeline Maestro de Entrenamiento Multi-Dataset (GPU)
### Fusión: Kaggle DFU + FUSeg Challenge (1200+ fotos) + DFUTissue (8 tejidos)

Este cuaderno permite:
1. Descargar y unificar los datasets clínicos abiertos de GitHub/Kaggle con tus datos previos.
2. Entrenar el modelo de visión clínica con aumento de datos avanzado (CLAHE, rotación, contraste).
3. Exportar el modelo optimizado a formato **ONNX** para que corra en milisegundos en la CPU de tu servidor Donweb sin costo de GPU.

In [22]:
# ── 1. INSTALACIÓN DE LIBRERÍAS CIENTÍFICAS ─────────────────────────────
!pip install -q timm onnx onnxruntime scikit-learn albumentations opencv-python matplotlib seaborn torchsummary

In [23]:
!pip install onnxscript

In [24]:
# ── 2. DESCARGA AUTOMÁTICA DE LOS DATASETS CLÍNICOS ────────────────────
import os
import shutil

!mkdir -p /content/datasets/fuseg
!mkdir -p /content/datasets/dfutissue

print('Descargando FUSeg Challenge (1.200+ imágenes con máscaras)...')
!git clone --depth 1 https://github.com/uwm-bigdata/wound-segmentation.git /content/tmp_fuseg
!cp -r /content/tmp_fuseg/data/* /content/datasets/fuseg/ 2>/dev/null || true
!rm -rf /content/tmp_fuseg

print('Descargando DFUTissueSegNet (Anotaciones de 8 tipos de tejido)...')
!git clone --depth 1 https://github.com/uwm-bigdata/DFUTissueSegNet.git /content/tmp_tissue
!cp -r /content/tmp_tissue/* /content/datasets/dfutissue/ 2>/dev/null || true
!rm -rf /content/tmp_tissue

print('✓ Datasets descargados y listos en /content/datasets/')
!ls -lh /content/datasets

Descargando FUSeg Challenge (1.200+ imágenes con máscaras)...
Cloning into '/content/tmp_fuseg'...
remote: Enumerating objects: 2557, done.
remote: Counting objects: 100% (2557/2557), done.
remote: Compressing objects: 100% (1753/1753), done.
remote: Total 2557 (delta 801), reused 2536 (delta 800), pack-reused 0 (from 0)
Receiving objects: 100% (2557/2557), 327.98 MiB | 12.87 MiB/s, done.
Resolving deltas: 100% (801/801), done.
Updating files: 100% (2593/2593), done.
Descargando DFUTissueSegNet (Anotaciones de 8 tipos de tejido)...
Cloning into '/content/tmp_tissue'...
remote: Enumerating objects: 1497, done.
remote: Counting objects: 100% (1497/1497), done.
remote: Compressing objects: 100% (1423/1423), done.
remote: Total 1497 (delta 63), reused 1471 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (1497/1497), 32.19 MiB | 19.76 MiB/s, done.
Resolving deltas: 100% (63/63), done.
✓ Datasets descargados y listos en /content/datasets/
total 8.0K
drwxr-xr-x 5 root root 4.0K Aug

In [25]:
# ── 3. MONTAJE DE GOOGLE DRIVE (OPCIONAL: PARA SUMAR TU DATASET PREVIO) ─
from google.colab import drive
drive.mount('/content/drive')

# Si tenés tu carpeta previa en Drive, podés enlazarla aquí:
DRIVE_DATASET_PATH = '/content/drive/MyDrive/DFU'
if os.path.exists(DRIVE_DATASET_PATH):
    print(f'✓ Dataset previo encontrado en Drive: {DRIVE_DATASET_PATH}')
else:
    print('ℹ️ Usando datasets públicos descargados de GitHub.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Dataset previo encontrado en Drive: /content/drive/MyDrive/DFU


In [26]:
# ── 4. ARQUITECTURA EFFICIENTNET-B0 + FINE-TUNING CLÍNICO ──────────────
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de entrenamiento: {device}')

class DFUClinicalClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(DFUClinicalClassifier, self).__init__()
        # Backbone EfficientNet-B0 (ultraliviano y de alta precisión)
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features

        # Cabeza de clasificación con Regularización Dropout para evitar sobreajuste
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out

model = DFUClinicalClassifier(num_classes=2).to(device)
print('✓ Modelo DFUClinicalClassifier inicializado correctamente.')

Dispositivo de entrenamiento: cuda
✓ Modelo DFUClinicalClassifier inicializado correctamente.


In [27]:
# ── 5. ENTRENAMIENTO CON DATA AUGMENTATION MÉDICA ──────────────────────
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

print('Listo para entrenar con hiperparámetros optimizados para heridas.')

Listo para entrenar con hiperparámetros optimizados para heridas.


In [ ]:
import onnxscript

# ── 6. EXPORTACIÓN AUTOMÁTICA A FORMATO ONNX (PARA EL SERVIDOR) ────────
model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_output_path = '/content/dfu_vision_v2.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_output_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f'🎉 ¡Modelo exportado exitosamente en: {onnx_output_path}!')
print('Tamaño del archivo ONNX listo para Donweb:')
!ls -lh /content/dfu_vision_v2.onnx

/tmp/ipykernel_1642/717309866.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0825 21:04:50.143000 1642 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DFUClinicalClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DFUClinicalClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
